# 🏗️ OneVoice Edge — Synthetic Noisy Construction Speech Generator
**Features:** Auto-Resume Checkpoint · Bulletproof Noise Repair · Python 3.12 + Colab Compatible

## Cell 1 — Mount Drive & Install

In [ ]:
import os
IN_COLAB = 'google.colab' in str(get_ipython())
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_ROOT = '/content/drive/MyDrive/onevoice_audio_v1'
else:
    OUTPUT_ROOT = '/kaggle/working/onevoice_audio_v1'
print(f'Output root: {OUTPUT_ROOT}')

In [ ]:
# nest_asyncio fixes asyncio.run() inside Jupyter/Colab event loop
!pip install -q edge-tts soundfile librosa pandas tqdm nest_asyncio

## Cell 2 — Clone Dataset

In [ ]:
if not os.path.exists('/content/OneVoice'):
    !git clone --depth 1 https://github.com/Platypus27-coder/OneVoice.git /content/OneVoice
else:
    print('Repo already cloned.')

if os.path.exists('/content/OneVoice/data/onevoice_construction_v2'):
    DATA_DIR = '/content/OneVoice/data/onevoice_construction_v2'
elif os.path.exists('/content/OneVoice/onevoice-edge/data/onevoice_construction_v2'):
    DATA_DIR = '/content/OneVoice/onevoice-edge/data/onevoice_construction_v2'
else:
    DATA_DIR = '/content/data/onevoice_construction_v2'

print('Data Path:', DATA_DIR)
print('Files:', os.listdir(DATA_DIR))

## Cell 3 — Configuration

In [ ]:
import random, json

NOISE_DIR   = os.path.join(OUTPUT_ROOT, 'noise_bank')
CLEAN_DIR   = os.path.join(OUTPUT_ROOT, 'clean')
NOISY_DIR   = os.path.join(OUTPUT_ROOT, 'noisy')
MANIFEST    = os.path.join(OUTPUT_ROOT, 'manifest.jsonl')

for d in [OUTPUT_ROOT, NOISE_DIR, CLEAN_DIR, NOISY_DIR]:
    os.makedirs(d, exist_ok=True)

SAMPLES_PER_TEXT = 2
SAMPLE_RATE      = 16000
MAX_UTTERANCES   = None  # set int to limit for quick test

VI_SPEAKERS = [
    ('vi-VN-HoaiMyNeural', None),  # Vietnamese Female
    ('vi-VN-NamMinhNeural', None), # Vietnamese Male
]
NOISE_CLASSES = [
    'excavator.wav', 'angle_grinder.wav', 'drilling.wav', 'hammer.wav',
    'diesel_engine.wav', 'generator.wav', 'truck.wav', 'wind.wav', 'worker_babble.wav',
]
SNR_OPTIONS = [0, 5, 10, 15, 20]
print('Config ready.')

## Cell 4 — Download & Auto-Repair Noise Bank

In [ ]:
import numpy as np
import soundfile as sf

ESC50_BASE = 'https://raw.githubusercontent.com/karolpiczak/ESC-50/master/audio'
NOISE_URLS = {
    'excavator.wav':     f'{ESC50_BASE}/1-116765-A-41.wav',
    'angle_grinder.wav': f'{ESC50_BASE}/3-156897-A-13.wav',
    'drilling.wav':      f'{ESC50_BASE}/4-182368-A-12.wav',
    'hammer.wav':        f'{ESC50_BASE}/3-149189-A-13.wav',
    'diesel_engine.wav': f'{ESC50_BASE}/1-26143-A-43.wav',
    'generator.wav':     f'{ESC50_BASE}/2-109371-A-43.wav',
    'truck.wav':         f'{ESC50_BASE}/5-219213-A-11.wav',
    'wind.wav':          f'{ESC50_BASE}/1-179701-A-25.wav',
    'worker_babble.wav': f'{ESC50_BASE}/1-26143-A-43.wav',
}

def generate_synthetic_noise(noise_type: str, duration_sec: int = 10, sr: int = 16000) -> np.ndarray:
    t = np.linspace(0, duration_sec, int(sr * duration_sec))
    if 'grinder' in noise_type or 'drill' in noise_type:
        noise = 0.6 * np.sin(2 * np.pi * 3200 * t + np.sin(2 * np.pi * 50 * t)) + 0.4 * np.random.normal(0, 1, len(t))
    elif any(k in noise_type for k in ('engine', 'excavator', 'generator', 'truck')):
        noise = 0.5 * np.sin(2 * np.pi * 60 * t) + 0.3 * np.sin(2 * np.pi * 120 * t) + 0.3 * np.random.normal(0, 1, len(t))
    elif 'hammer' in noise_type:
        noise = 0.2 * np.random.normal(0, 1, len(t))
        for idx in np.arange(0, len(t), int(sr * 0.8)):
            end = min(idx + int(sr * 0.05), len(t))
            noise[idx:end] += np.random.normal(0, 3, end - idx)
    else:
        noise = np.convolve(np.random.normal(0, 1, len(t)), [0.05, -0.09, 0.05], mode='same')
    return np.clip(noise / (np.max(np.abs(noise)) + 1e-9), -1.0, 1.0)

def is_valid_wav(path):
    if not os.path.exists(path) or os.path.getsize(path) < 10000:
        return False
    try:
        sf.read(path)
        return True
    except Exception:
        return False

def download_noise_bank():
    os.makedirs(NOISE_DIR, exist_ok=True)
    for fname, url in NOISE_URLS.items():
        dst = os.path.join(NOISE_DIR, fname)
        if is_valid_wav(dst):
            print(f'  ✅ {fname} OK')
            continue
        if os.path.exists(dst):
            os.remove(dst)
        print(f'Downloading {fname}...')
        os.system(f'wget -q -O "{dst}" "{url}"')
        if not is_valid_wav(dst):
            print(f'  ⚡ Generating synthetic noise for {fname}...')
            if os.path.exists(dst):
                os.remove(dst)
            sf.write(dst, generate_synthetic_noise(fname), 16000)
    print('\n✅ Noise bank ready!')

download_noise_bank()

## Cell 5 — Audio Functions (Colab-Compatible TTS)

In [ ]:
import numpy as np
import soundfile as sf
import librosa
import asyncio
import edge_tts
import nest_asyncio
nest_asyncio.apply()  # ← Fix asyncio.run() inside Jupyter/Colab

def tts_synthesize(text: str, out_path: str, voice_name: str, speaker=None) -> bool:
    try:
        loop = asyncio.get_event_loop()
        loop.run_until_complete(
            edge_tts.Communicate(text, voice_name).save(out_path)
        )
        return True
    except Exception as e:
        print(f'  [TTS ERR] {e}')
        return False

def apply_rir(speech: np.ndarray, sr: int, rir_path: str = None) -> np.ndarray:
    if rir_path and os.path.exists(rir_path):
        rir, _ = librosa.load(rir_path, sr=sr, mono=True)
        out = np.convolve(speech, rir, mode='full')[:len(speech)]
    else:
        delay = int(sr * random.uniform(0.03, 0.08))
        decay = random.uniform(0.2, 0.4)
        echo  = np.zeros_like(speech)
        echo[delay:] = speech[:-delay] * decay
        out = speech + echo
    return out / (np.max(np.abs(out)) + 1e-9)

def mix_noise(speech: np.ndarray, noise: np.ndarray, snr_db: float) -> np.ndarray:
    if len(noise) < len(speech):
        noise = np.tile(noise, int(np.ceil(len(speech) / len(noise))))
    noise = noise[random.randint(0, max(0, len(noise) - len(speech))): len(speech) + random.randint(0, max(0, len(noise) - len(speech)))]
    noise = noise[:len(speech)]
    rms_s = np.sqrt(np.mean(speech ** 2) + 1e-9)
    rms_n = np.sqrt(np.mean(noise  ** 2) + 1e-9)
    mixed = speech + (rms_s / (rms_n * (10 ** (snr_db / 20)))) * noise
    return np.clip(mixed / (np.max(np.abs(mixed)) + 1e-9), -1.0, 1.0)

def augment(audio: np.ndarray, gain_range=(-3.0, 3.0), clip_prob=0.05) -> np.ndarray:
    audio = audio * (10 ** (random.uniform(*gain_range) / 20))
    if random.random() < clip_prob:
        t = random.uniform(0.7, 0.95)
        audio = np.clip(audio, -t, t)
    return audio

print('✅ Audio functions ready (nest_asyncio applied)')

## Cell 6 — Generate Dataset (Auto-Resume + Checkpoint)

In [ ]:
import pandas as pd
from tqdm.notebook import tqdm

def generate_dataset():
    for d in [CLEAN_DIR, NOISY_DIR, OUTPUT_ROOT]:
        os.makedirs(d, exist_ok=True)

    # ── Load checkpoint ─────────────────────────────────────
    existing_audios = set()
    if os.path.exists(MANIFEST):
        with open(MANIFEST, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line:
                    try: existing_audios.add(json.loads(line).get('audio'))
                    except: pass
        print(f'🔄 Resuming: {len(existing_audios)} samples already done.')

    # ── Load utterances ─────────────────────────────────────
    df = pd.read_csv(os.path.join(DATA_DIR, 'utterances_all.csv'))
    if MAX_UTTERANCES:
        df = df.head(MAX_UTTERANCES)
    print(f'Utterances: {len(df)}')

    # ── Load noise with auto-repair ──────────────────────────
    noise_cache = {}
    for nc in NOISE_CLASSES:
        path = os.path.join(NOISE_DIR, nc)
        if os.path.exists(path):
            try:
                noise_cache[nc], _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
            except Exception as e:
                print(f'  Repairing {nc}...')
                synth = generate_synthetic_noise(nc)
                sf.write(path, synth, SAMPLE_RATE)
                noise_cache[nc] = synth
    if not noise_cache:
        noise_cache['silence.wav'] = np.zeros(SAMPLE_RATE)

    usable_noises = list(noise_cache.keys())
    sample_idx    = len(existing_audios)
    skipped       = 0

    # ── Main loop ───────────────────────────────────────────
    with open(MANIFEST, 'a', encoding='utf-8') as mf:
        for _, row in tqdm(df.iterrows(), total=len(df), desc='Generating'):
            utt_id  = str(row['utterance_id'])
            vi_text = str(row['vi'])
            en_text = str(row.get('en', ''))
            domain  = str(row.get('domain', 'unknown'))
            intent  = str(row.get('intent', 'unknown'))
            risk    = str(row.get('risk_level', 'unknown'))
            split   = str(row.get('split', 'train'))

            # Fast skip if already done
            if all(
                f'{utt_id}_n{v+1:02d}.wav' in existing_audios and
                os.path.exists(os.path.join(NOISY_DIR, f'{utt_id}_n{v+1:02d}.wav'))
                for v in range(SAMPLES_PER_TEXT)
            ):
                continue

            # TTS → clean wav
            voice, _ = random.choice(VI_SPEAKERS)
            clean_fname = f'{utt_id}_clean.wav'
            clean_path  = os.path.join(CLEAN_DIR, clean_fname)
            if not os.path.exists(clean_path):
                if not tts_synthesize(vi_text, clean_path, voice):
                    skipped += 1; continue
            try:
                clean_audio, _ = librosa.load(clean_path, sr=SAMPLE_RATE, mono=True)
            except:
                skipped += 1; continue

            reverbed = apply_rir(clean_audio, SAMPLE_RATE)

            for v in range(SAMPLES_PER_TEXT):
                noisy_fname = f'{utt_id}_n{v+1:02d}.wav'
                noisy_path  = os.path.join(NOISY_DIR, noisy_fname)
                if noisy_fname in existing_audios and os.path.exists(noisy_path):
                    continue

                noise_name = random.choice(usable_noises)
                snr_db     = random.choice(SNR_OPTIONS)
                use_reverb = random.random() > 0.35
                base       = reverbed if use_reverb else clean_audio
                mixed      = augment(mix_noise(base, noise_cache[noise_name], snr_db))

                sf.write(noisy_path, mixed, SAMPLE_RATE)

                entry = {
                    'audio': noisy_fname, 'clean_audio': clean_fname,
                    'text': vi_text, 'translation': en_text,
                    'domain': domain, 'intent': intent,
                    'risk_level': risk, 'split': split,
                    'speaker_id': voice,
                    'noise_type': noise_name.replace('.wav', ''),
                    'snr_db': snr_db, 'reverb': use_reverb,
                    'rir_id': 'simulated_echo' if use_reverb else 'none',
                    'synthetic_speech': True, 'synthetic_noise_mix': True,
                    'sample_rate': SAMPLE_RATE,
                }
                mf.write(json.dumps(entry, ensure_ascii=False) + '\n')
                mf.flush()
                existing_audios.add(noisy_fname)
                sample_idx += 1

    print(f'\n✅ Done! Total: {sample_idx} | Skipped: {skipped}')

generate_dataset()

## Cell 7 — Verify Stats

In [ ]:
import pandas as pd
entries = [json.loads(l) for l in open(MANIFEST, encoding='utf-8') if l.strip()]
df_m = pd.DataFrame(entries)
print(f'Total: {len(df_m)}')
for col in ['domain', 'noise_type', 'snr_db', 'split']:
    print(f'\nBy {col}:')
    print(df_m[col].value_counts().to_string())

## Cell 8 — Listen Test (Colab only)

In [ ]:
from IPython.display import Audio, display
s = random.choice(entries)
print(f"Text: {s['text']} | Noise: {s['noise_type']} | SNR: {s['snr_db']}dB")
print('Clean:')
display(Audio(os.path.join(CLEAN_DIR, s['clean_audio']), rate=SAMPLE_RATE))
print('Noisy:')
display(Audio(os.path.join(NOISY_DIR, s['audio']), rate=SAMPLE_RATE))